In [10]:
%run -i ../../python_scripts/nb_setup.py

GPU Available: False


#### Choose the dataset you want to use :
* must be data the model did not see at training
* first column : __y_true__ $\in \{0,1\}$ are the samples labels  
* second column : __y_pred__ are the predicted class of samples ($\in \{0,1\}$ too)
* third column : __kappa_f__ is confidence function for each sample, $\in \mathbb{R}$

In [7]:
sgp_df = pickle.load(open("sgp_set_cnn", "rb"))
print(
    "N =",
    sgp_df.shape[0],
    "Proportion of 1s: ",
    np.round(sgp_df.y_true.sum() / sgp_df.shape[0], 2),
)
sgp_df.head(3)

N = 40000 Proportion of 1s:  0.1


,y_true,y_pred,kappa
0,0.0,0.0,0.968736
1,0.0,1.0,0.870544
2,0.0,0.0,0.882295


In [ ]:
theta_min, theta_max = 0.5, 1  # Sn-independent grid
num_targets = 50  # number of targets r* to consider when drawing metric/coverage of metric/theta curves
num_seed = 50

Parallel runs (same operation for several seeds...)

In [12]:
def run_one_seed(
    s,
    metric="standard",
    metric_targets=metric_targets,
    mode="dicho",
):
    train_set, test_set = train_test_split(sgp_df, seed=s)
    results = sgp_at_targets(
        train_set,
        test_set,
        metric_targets=metric_targets,
        metric=metric,
        mode=mode,
        theta_min=theta_min,
        theta_max=theta_max,
    )
    if results.shape[0] > 0:
        return results.loc[results.metric_bound < results.test_metric].shape[0]
    return None  # nothing to record for this seed

### __0/1 risk__

In [13]:
metric_targets = np.linspace(0.03, 0.2, num=num_targets)

In [ ]:
raw = list(
    tqdm(
        Parallel(n_jobs=-1)(
            delayed(
                lambda seed: run_one_seed(
                    seed, metric="standard", metric_targets=metric_targets, mode="dicho"
                )
            )(s)
            for s in range(num_seed)
        ),
        total=num_seed,
    )
)
failures_01 = [x for x in raw if x is not None]

In [ ]:
print("failure rate:", sum(failures_01) / (num_seed * num_targets))

### __FP risk__

In [ ]:
metric_targets = np.linspace(0.03, 0.16, num=num_targets)

In [ ]:
raw = list(
    tqdm(
        Parallel(n_jobs=-1)(
            delayed(
                lambda seed: run_one_seed(
                    seed, metric="FP", metric_targets=metric_targets, mode="dicho"
                )
            )(s)
            for s in range(num_seed)
        ),
        total=num_seed,
    )
)
failures_fp = [x for x in raw if x is not None]

In [ ]:
print("failure rate:", sum(failures_fp) / (num_seed * num_targets))

### __FN risk__

In [ ]:
metric_targets = np.linspace(0.001, 0.04, num=num_targets)

In [ ]:
raw = list(
    tqdm(
        Parallel(n_jobs=-1)(
            delayed(
                lambda seed: run_one_seed(
                    seed, metric="FN", metric_targets=metric_targets, mode="dicho"
                )
            )(s)
            for s in range(num_seed)
        ),
        total=num_seed,
    )
)
failures_fn = [x for x in raw if x is not None]

In [ ]:
print("failure rate:", sum(failures_fn) / (num_seed * num_targets))

### __FPR__

In [ ]:
metric_targets = np.linspace(0.01, 0.2, num=num_targets)

In [ ]:
raw = list(
    tqdm(
        Parallel(n_jobs=-1)(
            delayed(
                lambda seed: run_one_seed(
                    seed, metric="FPR", metric_targets=metric_targets, mode="greedy"
                )
            )(s)
            for s in range(num_seed)
        ),
        total=num_seed,
    )
)
failures_fpr = [x for x in raw if x is not None]

In [ ]:
print("failure rate:", sum(failures_fpr) / (num_seed * num_targets))

### __FNR__

In [5]:
metric_targets = np.linspace(0.001, 0.04, num=num_targets)

In [ ]:
raw = list(
    tqdm(
        Parallel(n_jobs=-1)(
            delayed(
                lambda seed: run_one_seed(
                    seed, metric="FNR", metric_targets=metric_targets, mode="greedy"
                )
            )(s)
            for s in range(num_seed)
        ),
        total=num_seed,
    )
)
failures_fnr = [x for x in raw if x is not None]

In [ ]:
print("failure rate:", sum(failures_fnr) / (num_seed * num_targets))

0